# Preprocessing

## Imports

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from src.lib.connectors import Connectors

print("Imports successful")

Imports successful


## Connect and Load

In [2]:
s3 = Connectors.connect_s3()
df = Connectors.load_data(s3)

Access key loaded: AKIA6...GJWR
Secret key loaded: ********************
Region: us-east-1
Bucket: card-fraud-detection-01

Verifying AWS connection...
Connected to AWS
Account ID: 977237814864
User: arn:aws:iam::977237814864:root
Local cache found — loading from disk...
Loaded in 0.5s


## Scaling

In [3]:
scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# Drop the originals
df = df.drop(columns=['Amount', 'Time'])

print("Amount and Time scaled")
print(f"Shape: {df.shape}")

Amount and Time scaled
Shape: (284807, 31)


## Trian/Test Split

In [4]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keeps fraud ratio consistent in both splits
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"\nTrain fraud rate: {y_train.mean()*100:.3f}%")
print(f"Test fraud rate: {y_test.mean()*100:.3f}%")

Train size: (227845, 30)
Test size: (56962, 30)

Train fraud rate: 0.173%
Test fraud rate: 0.172%


## Apply SMOTE

In [5]:
print("Applying SMOTE to training set...")
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}")
print(f"New train size: {X_train_sm.shape}")

Applying SMOTE to training set...
Before SMOTE: {0: 227451, 1: 394}
After SMOTE: {0: 227451, 1: 227451}
New train size: (454902, 30)


## Save to AWS

In [6]:
print("Saving processed data to S3...")


Connectors.save_to_s3(X_train_sm, s3, 'processed-data/X_train.pkl')
Connectors.save_to_s3(y_train_sm, s3, 'processed-data/y_train.pkl')
Connectors.save_to_s3(X_test, s3,     'processed-data/X_test.pkl')
Connectors.save_to_s3(y_test, s3,     'processed-data/y_test.pkl')

print("\nAll processed data saved to S3")

Saving processed data to S3...
Saving processed-data/X_train.pkl...


Uploading X_train.pkl: 100%|██████████████████████████████| 104M/104M [01:13<00:00, 1.49MB/s] 


Saved: processed-data/X_train.pkl
Saving processed-data/y_train.pkl...


Uploading y_train.pkl: 100%|██████████████████████████████| 3.47M/3.47M [00:08<00:00, 417kB/s]


Saved: processed-data/y_train.pkl
Saving processed-data/X_test.pkl...


Uploading X_test.pkl: 100%|██████████████████████████████| 13.5M/13.5M [00:23<00:00, 614kB/s] 


Saved: processed-data/X_test.pkl
Saving processed-data/y_test.pkl...


Uploading y_test.pkl: 100%|██████████████████████████████| 1.30M/1.30M [00:02<00:00, 497kB/s]

Saved: processed-data/y_test.pkl

All processed data saved to S3
